### Feature Engineering

In [78]:
import pandas as pd
import numpy as np

In [79]:
df = pd.read_csv("../data/processed/personalized_learning_processed.csv")

In [80]:
df.head()

,course_id,name,category,difficulty,estimated_hours,goal_role,prerequisites,skills,concept_overlap_group,module_list,...,target_roles,career_importance,learning_outcomes,practical_application,assessment_method,difficulty_score,career_value_score,skill_growth_score,num_prerequisites,num_skills
0,C001,"Anatomy & Physiology: Regulation, Integration,...",Health,Intermediate,28,Data Scientist,Medical Terminology,"Medical Terminology, Endocrinology, Life Scien...",5,Not Specified,...,Data Scientist,Important,Apply None concepts to practical tasks.,Coding Exercises,Coding Assignment,3,8,3.65,1,8
1,C002,Excel Essentials and Beyond,Data Science,Intermediate,49,Full Stack,Data Presentation,"Data Presentation, Pivot Tables And Charts, Sp...",64,"Navigate Excel with confidence, leveraging ess...",...,Full Stack,Important,Apply None concepts to practical tasks.,Coding Exercises,Coding Assignment,3,8,5.67,1,12
2,C003,Project Execution: Running the Project,Business,Beginner,8,AI Engineer,Continuous Improvement Process,"Continuous Improvement Process, Data Storytell...",45,Implement the key quality management concepts ...,...,AI Engineer,Important,Apply None concepts to practical tasks.,Coding Exercises,Coding Assignment,1,6,4.64,1,17
3,C004,Taxation of Multinationals for Everyone,Business,Intermediate,26,ML Engineer,Income Tax,"Income Tax, Tax Planning, Tax, Corporate Tax, ...",74,Participants will gain a comprehensive underst...,...,ML Engineer,Important,Apply None concepts to practical tasks.,Coding Exercises,Coding Assignment,3,8,3.25,1,7
4,C005,Creating Change through Social Entrepreneurship,Business,Intermediate,53,Backend Developer,Advocacy,"Advocacy, Entrepreneurship, Design Thinking, R...",58,Not Specified,...,Backend Developer,Important,Apply None concepts to practical tasks.,Coding Exercises,Coding Assignment,3,8,6.46,1,14


In [81]:
drop_cols = [
    "course_id",
    "url"
]

df = df.drop(columns=drop_cols)

In [82]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=500)

skills_matrix = tfidf.fit_transform(df["skills"])

In [83]:
df = pd.get_dummies(
    df,
    columns=[
        "difficulty",
        "category",
        "goal_role"
    ]
)

In [84]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["difficulty"] = le.fit_transform(df["difficulty_score"])

In [85]:
print(df.columns.tolist())

['name', 'estimated_hours', 'prerequisites', 'skills', 'concept_overlap_group', 'module_list', 'course_description', 'course_type', 'industry_relevance', 'job_market_demand', 'skill_level', 'skill_category', 'core_skills', 'prerequisite_level', 'recommended_background', 'next_courses', 'target_roles', 'career_importance', 'learning_outcomes', 'practical_application', 'assessment_method', 'difficulty_score', 'career_value_score', 'skill_growth_score', 'num_prerequisites', 'num_skills', 'difficulty_Advanced', 'difficulty_Beginner', 'difficulty_Intermediate', 'category_Arts And Humanities', 'category_Business', 'category_Computer Science', 'category_Data Science', 'category_Health', 'category_Information Technology', 'category_Language Learning', 'category_Math And Logic', 'category_Personal Development', 'category_Physical Science And Engineering', 'category_Social Sciences', 'goal_role_AI Engineer', 'goal_role_Backend Developer', 'goal_role_Data Analyst', 'goal_role_Data Scientist', 'go

In [86]:
df["learning_load"] = (
    df["estimated_hours"] * np.log1p(df["num_skills"])
).round(2)

In [87]:
df["skill_density"] = (
    df["num_skills"] /
    df["estimated_hours"]
).round(3)

In [88]:
df["prerequisite_density"] = (
    df["num_prerequisites"] /
    df["estimated_hours"]
).round(3)

In [89]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

df[["hours_scaled", "skills_scaled", "prereq_scaled"]] = scaler.fit_transform(
    df[["estimated_hours", "num_skills", "num_prerequisites"]]
)

df["learning_complexity"] = (
    0.4 * (df["difficulty_score"] / 5) +
    0.3 * df["hours_scaled"] +
    0.2 * df["skills_scaled"] +
    0.1 * df["prereq_scaled"]
)

df["learning_complexity"] = (df["learning_complexity"] * 100).round(2)

In [90]:
df["estimated_hours_scaled"] = scaler.fit_transform(
    df[["estimated_hours"]]
)

In [91]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

num_cols = [
    "estimated_hours",
    "num_prerequisites",
    "num_skills",
    "learning_load",
    "skill_density"
]

df[num_cols] = scaler.fit_transform(df[num_cols])

In [92]:
df.drop(
    columns=[
        "career_value_score",
        "skill_growth_score"
    ],
    inplace=True
)

In [93]:
corr = df.corr(numeric_only=True)

In [94]:
df.to_csv(
    "../data/processed/personalized_learning_features.csv",
    index=False
)

In [95]:
df.head()

,name,estimated_hours,prerequisites,skills,concept_overlap_group,module_list,course_description,course_type,industry_relevance,job_market_demand,...,goal_role_ML Engineer,difficulty,learning_load,skill_density,prerequisite_density,hours_scaled,skills_scaled,prereq_scaled,learning_complexity,estimated_hours_scaled
0,"Anatomy & Physiology: Regulation, Integration,...",-0.456507,Medical Terminology,"Medical Terminology, Endocrinology, Life Scien...",5,Not Specified,Introduces key concepts and practical applicat...,Health,High,High,...,False,1,-0.638737,-0.481329,0.036,0.384615,0.333333,0.0,42.21,0.384615
1,Excel Essentials and Beyond,0.894994,Data Presentation,"Data Presentation, Pivot Tables And Charts, Sp...",64,"Navigate Excel with confidence, leveraging ess...",Introduces key concepts and practical applicat...,Data Science,High,High,...,False,1,0.947892,-0.603006,0.020,0.788462,0.523810,0.0,58.13,0.788462
2,Project Execution: Running the Project,-1.743651,Continuous Improvement Process,"Continuous Improvement Process, Data Storytell...",45,Implement the key quality management concepts ...,Introduces key concepts and practical applicat...,Business,High,High,...,False,0,-1.588340,4.976354,0.125,0.000000,0.761905,0.0,23.24,0.000000
3,Taxation of Multinationals for Everyone,-0.585222,Income Tax,"Income Tax, Tax Planning, Tax, Corporate Tax, ...",74,Participants will gain a comprehensive underst...,Introduces key concepts and practical applicat...,Business,High,High,...,True,1,-0.822970,-0.531780,0.038,0.346154,0.285714,0.0,40.10,0.346154
4,Creating Change through Social Entrepreneurship,1.152423,Advocacy,"Advocacy, Entrepreneurship, Design Thinking, R...",58,Not Specified,Introduces key concepts and practical applicat...,Business,High,High,...,False,1,1.389309,-0.546619,0.019,0.865385,0.619048,0.0,62.34,0.865385
